# Module 3: Temporal Difference Learning
## Notebook 2: SARSA vs. Q-Learning

**Learning Objectives**

By completing this notebook, you will:
1. Implement SARSA (on-policy TD control) and Q-learning (off-policy TD control) from scratch
2. Apply both algorithms to the Cliff Walking environment and observe their distinct behaviors
3. Explain why SARSA learns a safe path while Q-learning learns the optimal but risky path
4. Connect the on-policy vs. off-policy distinction to real-world deployment concerns

**Prerequisites**
- TD(0) prediction (Notebook 1)
- Epsilon-greedy exploration
- The concept of on-policy vs. off-policy learning

**Estimated Time: 14 minutes**

---

### The Central Difference

Both SARSA and Q-learning update a Q-function $Q(s, a)$. They differ in **which next action** they use for bootstrapping:

| Algorithm | Update target | Policy |
|-----------|--------------|--------|
| SARSA | $R + \gamma Q(S', A')$ where $A'$ is the action **actually taken** | On-policy |
| Q-learning | $R + \gamma \max_{a'} Q(S', a')$ regardless of what is taken | Off-policy |

This one difference leads to dramatically different behavior when exploration (epsilon-greedy) is still active during training.

In [ ]:
# Purpose: Import dependencies and set seeds
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Setup complete.")

## 1. Cliff Walking Environment

Cliff Walking is a gridworld from Sutton & Barto (Example 6.6). The agent navigates a 4x12 grid:
- **Start**: bottom-left corner
- **Goal**: bottom-right corner
- **Cliff**: the bottom row between start and goal
- **Reward**: -1 per step, -100 for stepping on the cliff (and returning to start)

```
 ___________________________________________
|   |   |   |   |   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |   |   |   |   |
| S | C | C | C | C | C | C | C | C | C | C | G |
 -------------------------------------------
```

The safe path goes up from S, across the top rows, and back down to G. The risky path goes along the bottom right next to the cliff.

In [ ]:
# Purpose: Implement the Cliff Walking gridworld
# Key Concept: The -100 cliff reward creates a meaningful risk-vs-reward tradeoff

class CliffWalking:
    """
    4x12 Cliff Walking environment.
    Actions: 0=Up, 1=Right, 2=Down, 3=Left
    """

    ROWS = 4
    COLS = 12
    N_ACTIONS = 4

    # Action deltas: (row_delta, col_delta)
    ACTION_DELTAS = {
        0: (-1, 0),   # Up
        1: (0, +1),   # Right
        2: (+1, 0),   # Down
        3: (0, -1),   # Left
    }
    ACTION_NAMES = ['Up', 'Right', 'Down', 'Left']

    def __init__(self):
        self.start = (self.ROWS - 1, 0)           # Bottom-left
        self.goal  = (self.ROWS - 1, self.COLS - 1)  # Bottom-right
        # Cliff: bottom row, columns 1..10
        self.cliff = {(self.ROWS - 1, c) for c in range(1, self.COLS - 1)}

    def reset(self):
        """Return start state as (row, col)."""
        return self.start

    def step(self, state, action):
        """
        Take action from state.

        Returns
        -------
        next_state : tuple (row, col)
        reward     : float
        done       : bool
        """
        dr, dc = self.ACTION_DELTAS[action]
        r, c = state

        # Clamp to grid boundaries
        next_r = max(0, min(self.ROWS - 1, r + dr))
        next_c = max(0, min(self.COLS - 1, c + dc))
        next_state = (next_r, next_c)

        if next_state in self.cliff:
            # Fall off cliff: big penalty, return to start
            return self.start, -100.0, False
        elif next_state == self.goal:
            return next_state, -1.0, True
        else:
            return next_state, -1.0, False

    def state_to_idx(self, state):
        """Convert (row, col) to flat index for Q-table lookup."""
        return state[0] * self.COLS + state[1]

    @property
    def n_states(self):
        return self.ROWS * self.COLS


env = CliffWalking()
print(f"Grid size: {env.ROWS}x{env.COLS} = {env.n_states} states")
print(f"Start: {env.start}, Goal: {env.goal}")
print(f"Cliff cells: {len(env.cliff)} cells")

# Quick sanity check: step right from start falls on cliff
ns, r, done = env.step(env.start, 1)  # action 1 = Right
print(f"Step Right from start: next={ns}, reward={r}, done={done}")

## 2. SARSA: On-Policy TD Control

SARSA updates the Q-value of the **(S, A)** pair using the **actual next action $A'$** chosen by the epsilon-greedy policy:

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma Q(S', A') - Q(S, A) \right]$$

The name comes from the quintuple $(S, A, R, S', A')$. Because the update includes the exploratory action $A'$, SARSA **accounts for the risk of exploration** and learns to stay away from the cliff.

In [ ]:
# Purpose: Implement SARSA (on-policy TD control)
# Key Concept: The target uses the next action actually taken under epsilon-greedy

def epsilon_greedy(Q_row, epsilon, n_actions, rng):
    """
    Select an action using epsilon-greedy policy.

    Parameters
    ----------
    Q_row    : np.ndarray, shape (n_actions,) — Q-values for current state
    epsilon  : float — exploration probability
    n_actions: int
    rng      : np.random.Generator

    Returns
    -------
    int — selected action
    """
    if rng.random() < epsilon:
        return rng.integers(n_actions)   # explore
    else:
        return int(np.argmax(Q_row))     # exploit


def sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=42):
    """
    SARSA (on-policy TD control) for CliffWalking.

    Returns
    -------
    Q              : np.ndarray, shape (n_states, n_actions)
    episode_rewards: list — sum of rewards per episode
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.N_ACTIONS))
    episode_rewards = []

    for _ in range(n_episodes):
        state = env.reset()
        s_idx = env.state_to_idx(state)

        # Step 1: Choose initial action with epsilon-greedy
        action = epsilon_greedy(Q[s_idx], epsilon, env.N_ACTIONS, rng)

        total_reward = 0.0
        done = False

        while not done:
            # Step 2: Take action, observe result
            next_state, reward, done = env.step(state, action)
            ns_idx = env.state_to_idx(next_state)
            total_reward += reward

            # Step 3: Choose next action with epsilon-greedy (ON-POLICY)
            next_action = epsilon_greedy(Q[ns_idx], epsilon, env.N_ACTIONS, rng)

            # Step 4: SARSA update — uses next_action (the actual next action)
            td_error = reward + gamma * Q[ns_idx, next_action] - Q[s_idx, action]
            Q[s_idx, action] += alpha * td_error

            state  = next_state
            s_idx  = ns_idx
            action = next_action   # reuse the chosen next action

        episode_rewards.append(total_reward)

    return Q, episode_rewards


Q_sarsa, rewards_sarsa = sarsa(env, n_episodes=500, alpha=0.5, epsilon=0.1)
print(f"SARSA — mean reward last 50 episodes: {np.mean(rewards_sarsa[-50:]):.1f}")

## 3. Q-Learning: Off-Policy TD Control

Q-learning uses the **maximum Q-value over all possible next actions**, regardless of which action will actually be taken:

$$Q(S, A) \leftarrow Q(S, A) + \alpha \left[ R + \gamma \max_{a'} Q(S', a') - Q(S, A) \right]$$

This makes Q-learning **off-policy**: the behavior policy (epsilon-greedy) is different from the target policy (greedy). Q-learning converges to the optimal Q-function $Q^*$, but ignoring exploration risk means it discovers the path right next to the cliff.

In [ ]:
# Purpose: Implement Q-learning (off-policy TD control)
# Key Concept: max over next actions makes this off-policy — it targets Q*

def q_learning(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=42):
    """
    Q-learning (off-policy TD control) for CliffWalking.

    Returns
    -------
    Q              : np.ndarray, shape (n_states, n_actions)
    episode_rewards: list — sum of rewards per episode
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.N_ACTIONS))
    episode_rewards = []

    for _ in range(n_episodes):
        state = env.reset()
        s_idx = env.state_to_idx(state)
        total_reward = 0.0
        done = False

        while not done:
            # Step 1: Choose action with epsilon-greedy (behavior policy)
            action = epsilon_greedy(Q[s_idx], epsilon, env.N_ACTIONS, rng)

            # Step 2: Take action, observe result
            next_state, reward, done = env.step(state, action)
            ns_idx = env.state_to_idx(next_state)
            total_reward += reward

            # Step 3: Q-learning update — uses max over next actions (OFF-POLICY)
            best_next_q = np.max(Q[ns_idx])   # greedy target
            td_error = reward + gamma * best_next_q - Q[s_idx, action]
            Q[s_idx, action] += alpha * td_error

            state = next_state
            s_idx = ns_idx

        episode_rewards.append(total_reward)

    return Q, episode_rewards


Q_ql, rewards_ql = q_learning(env, n_episodes=500, alpha=0.5, epsilon=0.1)
print(f"Q-learning — mean reward last 50 episodes: {np.mean(rewards_ql[-50:]):.1f}")

## 4. Visualizing Learned Paths

We extract the greedy path from each learned Q-table and draw it on the grid. This makes the behavioral difference concrete.

In [ ]:
# Purpose: Extract the greedy path from a Q-table and visualize it
# Key Concept: Greedy path shows what the policy does without exploration

def extract_greedy_path(env, Q, max_steps=100):
    """
    Follow the greedy policy from start to goal (or max_steps).

    Returns
    -------
    path : list of (row, col) tuples
    """
    state = env.reset()
    path = [state]
    visited = {state}

    for _ in range(max_steps):
        s_idx = env.state_to_idx(state)
        action = int(np.argmax(Q[s_idx]))
        next_state, _, done = env.step(state, action)
        path.append(next_state)

        if done or next_state in visited:
            break
        visited.add(next_state)
        state = next_state

    return path


def plot_paths(env, Q_sarsa, Q_ql):
    path_sarsa = extract_greedy_path(env, Q_sarsa)
    path_ql    = extract_greedy_path(env, Q_ql)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for ax, Q, path, title, color in [
        (axes[0], Q_sarsa, path_sarsa, 'SARSA (Safe Path)', 'steelblue'),
        (axes[1], Q_ql,    path_ql,    'Q-Learning (Optimal Path)', 'tomato'),
    ]:
        # Draw grid
        grid = np.zeros((env.ROWS, env.COLS))
        for (r, c) in env.cliff:
            grid[r, c] = -1    # cliff
        gr, gc = env.goal
        grid[gr, gc] = 2       # goal

        cmap = ListedColormap(['#e74c3c', 'white', '#2ecc71'])
        ax.imshow(grid, cmap=cmap, vmin=-1, vmax=2, aspect='auto')

        # Draw path
        rs = [p[0] for p in path]
        cs = [p[1] for p in path]
        ax.plot(cs, rs, '-o', color=color, linewidth=2.5, markersize=6,
                label='Greedy path', zorder=5)

        # Annotate start
        sr, sc = env.start
        ax.text(sc, sr, 'S', ha='center', va='center', fontsize=14,
                fontweight='bold', color='navy')
        ax.text(gc, gr, 'G', ha='center', va='center', fontsize=14,
                fontweight='bold', color='darkgreen')

        ax.set_title(title, fontsize=13)
        ax.set_xticks([])
        ax.set_yticks([])

        # Add cliff label
        ax.text(5.5, env.ROWS - 1, 'CLIFF', ha='center', va='center',
                fontsize=11, color='white', fontweight='bold')

    plt.suptitle('Greedy Paths After Training (epsilon=0.1, 500 episodes)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../resources/sarsa_vs_qlearning_paths.png', dpi=120, bbox_inches='tight')
    plt.show()


plot_paths(env, Q_sarsa, Q_ql)

## 5. Learning Curves

The reward curves during training tell a different story: Q-learning's optimal policy is riskier during training (exploration sometimes causes cliff falls), so its average rewards per episode are lower than SARSA's during the training phase.

In [ ]:
# Purpose: Plot training reward curves for SARSA and Q-learning
# Key Concept: Q-learning has lower rewards DURING training despite learning the optimal policy

def smooth(data, window=20):
    """Simple moving average smoothing."""
    return np.convolve(data, np.ones(window) / window, mode='valid')


fig, ax = plt.subplots(figsize=(12, 5))

# Raw rewards (faint)
ax.plot(rewards_sarsa, color='steelblue', alpha=0.2, linewidth=0.8)
ax.plot(rewards_ql,    color='tomato',    alpha=0.2, linewidth=0.8)

# Smoothed
ax.plot(smooth(rewards_sarsa), color='steelblue', linewidth=2,
        label=f'SARSA (smoothed, final avg={np.mean(rewards_sarsa[-50:]):.0f})')
ax.plot(smooth(rewards_ql),    color='tomato',    linewidth=2,
        label=f'Q-learning (smoothed, final avg={np.mean(rewards_ql[-50:]):.0f})')

ax.axhline(-13, color='green', linestyle='--', linewidth=1.5,
           label='Optimal policy return (-13)')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward', fontsize=12)
ax.set_title('SARSA vs. Q-Learning: Training Rewards on Cliff Walking', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-500, 10)

plt.tight_layout()
plt.savefig('../resources/sarsa_vs_qlearning_rewards.png', dpi=120, bbox_inches='tight')
plt.show()

print("SARSA: safer DURING training (explorations don't go near cliff)")
print("Q-learning: learns optimal path but pays the price during exploration")

### Exercise 3.2.1: Effect of Epsilon on the Behavioral Difference

**Task:** The gap between SARSA and Q-learning during training is caused by epsilon (exploration). Run both algorithms with epsilon values [0.01, 0.1, 0.3] and plot the mean episode reward over the last 50 episodes for each. Which epsilon causes the biggest gap between SARSA and Q-learning?

**Expected Output:** A bar chart or line plot showing SARSA vs Q-learning performance at each epsilon value.

<details>
<summary>Hint 1</summary>
Loop over epsilons, run both algorithms, and collect np.mean(rewards[-50:]) for each.
</details>

<details>
<summary>Hint 2</summary>
As epsilon increases, SARSA increasingly accounts for exploration risk, widening the performance gap during training.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

epsilons = [0.01, 0.1, 0.3]
sarsa_means = []
ql_means    = []

for eps in epsilons:
    _, r_sarsa = sarsa(env, n_episodes=500, alpha=0.5, epsilon=eps, seed=0)
    _, r_ql    = q_learning(env, n_episodes=500, alpha=0.5, epsilon=eps, seed=0)
    sarsa_means.append(np.mean(r_sarsa[-50:]))
    ql_means.append(np.mean(r_ql[-50:]))

x = np.arange(len(epsilons))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, sarsa_means, width, label='SARSA',     color='steelblue')
bars2 = ax.bar(x + width/2, ql_means,    width, label='Q-learning', color='tomato')
ax.set_xticks(x)
ax.set_xticklabels([f'epsilon={e}' for e in epsilons], fontsize=11)
ax.set_ylabel('Mean Episode Reward (last 50 eps)', fontsize=12)
ax.set_title('Effect of Epsilon on SARSA vs. Q-Learning Performance', fontsize=13)
ax.legend(fontsize=11)

# Annotate bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 5,
            f'{bar.get_height():.0f}', ha='center', va='top', fontsize=10, color='white')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 5,
            f'{bar.get_height():.0f}', ha='center', va='top', fontsize=10, color='white')

plt.tight_layout()
plt.show()

for eps, sm, qm in zip(epsilons, sarsa_means, ql_means):
    print(f"epsilon={eps}: SARSA={sm:.1f}, Q-learning={qm:.1f}, gap={sm-qm:.1f}")

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_321():
    assert len(sarsa_means) == 3 and len(ql_means) == 3, \
        "Run all 3 epsilon values for both algorithms"
    # At epsilon=0.3, SARSA should be safer than Q-learning
    idx_03 = epsilons.index(0.3)
    assert sarsa_means[idx_03] > ql_means[idx_03], \
        f"At epsilon=0.3, SARSA should outperform Q-learning during training. Got SARSA={sarsa_means[idx_03]:.1f}, QL={ql_means[idx_03]:.1f}"
    print("Exercise 3.2.1 passed!")

test_exercise_321()

### Exercise 3.2.2: Q-Table Value Inspection

**Task:** Extract and visualize the learned value function $V(s) = \max_a Q(s, a)$ for both SARSA and Q-learning as a heatmap over the grid. Exclude terminal states.

<details>
<summary>Hint 1</summary>
For each state index i, compute max_a Q[i, :] to get V(s). Reshape to (ROWS, COLS) for imshow.
</details>

<details>
<summary>Hint 2</summary>
Use plt.imshow with cmap='RdYlGn' to get red=bad, green=good coloring. Add plt.colorbar().
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

for ax, Q, title in [
    (axes[0], Q_sarsa, 'SARSA: V(s) = max_a Q(s,a)'),
    (axes[1], Q_ql,    'Q-learning: V(s) = max_a Q(s,a)'),
]:
    # Compute V(s) = max_a Q(s, a) for every state
    V = np.max(Q, axis=1).reshape(env.ROWS, env.COLS)

    # Mask cliff cells (they have Q values from cliff transitions)
    masked = np.ma.array(V)
    for (r, c) in env.cliff:
        masked[r, c] = np.ma.masked

    im = ax.imshow(masked, cmap='RdYlGn', aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Annotate start and goal
    sr, sc = env.start
    gr, gc = env.goal
    ax.text(sc, sr, 'S', ha='center', va='center', fontsize=14, fontweight='bold')
    ax.text(gc, gr, 'G', ha='center', va='center', fontsize=14, fontweight='bold')

    ax.set_title(title, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Learned Value Functions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_322():
    V_sarsa = np.max(Q_sarsa, axis=1).reshape(env.ROWS, env.COLS)
    V_ql    = np.max(Q_ql,    axis=1).reshape(env.ROWS, env.COLS)
    assert V_sarsa.shape == (env.ROWS, env.COLS), \
        f"Expected shape ({env.ROWS}, {env.COLS}), got {V_sarsa.shape}"
    # Top row states should have higher values in SARSA (safe path goes up)
    top_row_sarsa = np.mean(V_sarsa[0, 1:-1])
    bottom_row_sarsa = np.mean(V_sarsa[2, 1:-1])
    # Either top or bottom of interior rows is fine; just check it's computed
    assert V_ql.shape == (env.ROWS, env.COLS), \
        f"Expected shape ({env.ROWS}, {env.COLS}), got {V_ql.shape}"
    print("Exercise 3.2.2 passed!")
    print(f"SARSA top-row avg value: {top_row_sarsa:.2f}")
    print(f"SARSA bottom-interior avg value: {bottom_row_sarsa:.2f}")

test_exercise_322()

## Key Takeaways

1. **SARSA is on-policy**: it updates Q using the action actually taken, so it accounts for exploration risk. This leads to learning the safe path on Cliff Walking.
2. **Q-learning is off-policy**: it updates Q using the greedy action, targeting $Q^*$ directly. The optimal (risky) path is learned, but exploration causes cliff falls during training.
3. **Neither is universally better**: SARSA is preferred when safe behavior during training matters (robotics, industrial control). Q-learning is preferred when sample efficiency and optimality are paramount.
4. **Epsilon matters enormously**: as epsilon increases, SARSA becomes increasingly conservative and Q-learning increasingly reckless during training. At epsilon=0, both converge to the same policy.
5. **The SARSA name**: comes from the tuple $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$ — all five elements used in the update.

## What's Next

Notebook 3 adds **Expected SARSA** to the comparison and runs a systematic benchmark of all four TD algorithms (TD(0), SARSA, Q-learning, Expected SARSA) with confidence bands, giving you a clearer picture of their relative sample efficiency.